# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [ ]:
def get_embedding(text):
    result = client.models.embed_content(
        # model='text-embedding-004',  # sesuaikan versi model embedding
        model ='gemini-embedding-2', # Model 1
        # model = 'text-embedding-001' # Model 2
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [6]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


In [23]:
dimension

3072

In [24]:
index

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x0000027B7EA320B0> >

In [25]:
import numpy as np

# Mengambil seluruh vektor dari index
vectors = index.reconstruct_n(0, index.ntotal)

print("Bentuk data:", vectors.shape)
print("\nVektor pertama:")
print(vectors[0])

Bentuk data: (59, 3072)

Vektor pertama:
[-0.00509918  0.02026477 -0.01515723 ...  0.01789909 -0.00673305
  0.00923708]


In [27]:
import pandas as pd

df_embeddings = pd.DataFrame(vectors)

display(df_embeddings.head())

,0,1,2,3,4,5,6,7,8,9,...,3062,3063,3064,3065,3066,3067,3068,3069,3070,3071
0,-0.005099,0.020265,-0.015157,-0.026772,-0.014033,0.014844,0.019630,-0.019880,0.004624,-0.046713,...,-0.115710,0.011979,-0.049737,-0.024333,0.002349,-0.024551,0.014671,0.017899,-0.006733,0.009237
1,0.019191,0.004906,-0.017830,-0.013257,0.012089,0.008484,0.024918,-0.026073,0.000369,-0.036230,...,-0.109741,-0.002827,-0.041439,-0.029958,0.006741,-0.011784,0.013008,-0.011008,0.024976,0.003983
2,0.021523,0.032422,-0.012614,-0.007288,-0.003644,0.025168,0.012428,-0.009822,-0.016135,-0.049532,...,-0.107568,0.005964,-0.047115,-0.023192,-0.003278,-0.027070,0.000450,-0.008415,0.001254,-0.004772
3,0.023012,0.025904,-0.001143,0.002040,-0.013037,0.026505,-0.002209,-0.012417,-0.007438,-0.050893,...,-0.117611,-0.008303,-0.048140,-0.012441,-0.011568,-0.015088,0.014385,-0.004255,0.002801,-0.017052
4,0.013253,0.009533,-0.019920,-0.029963,-0.000629,0.027137,0.011794,-0.006078,-0.009726,-0.043445,...,-0.102801,-0.003180,-0.042227,-0.011614,-0.007402,-0.018645,-0.004520,-0.011430,0.012809,-0.005998


In [ ]:
import pandas as pd

print("Jumlah entri:", len(df))
print("\nKolom yang tersedia:")
print(df.columns.tolist())

print("\nInformasi struktur data:")
df.info()

if "category" in df.columns:
    print("\nJumlah entri per kategori:")
    print(df["category"].value_counts(dropna=False))

elif "kategori" in df.columns:
    print("\nJumlah entri per kategori:")
    print(df["kategori"].value_counts(dropna=False))

print("\nContoh data:")
display(df.head())

Jumlah entri: 59

Kolom yang tersedia:
['faq_id', 'category', 'question', 'answer', 'content']

Informasi struktur data:
<class 'pandas.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   faq_id    59 non-null     int64
 1   category  59 non-null     str  
 2   question  59 non-null     str  
 3   answer    59 non-null     str  
 4   content   59 non-null     str  
dtypes: int64(1), str(4)
memory usage: 2.4 KB

Jumlah entri per kategori:
category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64

Contoh data:


,faq_id,category,question,answer,content
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ...",Q: Bagaimana cara melakukan pemesanan di platf...
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...,Q: Apakah saya bisa memesan tanpa membuat akun...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...,Q: Bagaimana cara membatalkan pesanan yang sud...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...,Q: Berapa lama waktu yang dibutuhkan sebelum p...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...,Q: Apakah saya bisa mengubah alamat pengiriman...


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [7]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.6173067)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.61870193)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.634567)}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [8]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

In [ ]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [19]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        # model='gemini-3.5-flash',  # sesuaikan versi model
        model = 'gemini-3.5-flash-lite', # Model 2
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Halo! Tentu, saya siap membantu.

Terkait pertanyaan Anda, fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank. 

Ada hal lain yang ingin Anda tanyakan?

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [20]:
# TEST QUERIES 2

test_queries = [
    # TODO: isi minimal 5 query uji coba di sini
    "Bagaimana cara mengetahui apakah suatu produk asli atau palsu?",
    "Bagaimana jika penjual menolak pengajuan pengembalian saya?",
    "Saya memiliki voucher, bagaimana cara menggunakannya?",
    "Cara mengganti password akun saya yang lupa?",
    "Apa itu reverse engineering dan bagaimana cara kerjanya?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: Bagaimana cara mengetahui apakah suatu produk asli atau palsu?
Jawaban: Halo! Untuk mengetahui apakah suatu produk asli atau palsu, kamu bisa berbelanja di toko resmi atau terverifikasi, serta memeriksa deskripsi dan ulasan dari pembeli lain sebagai indikator keaslian produk. 

Ada lagi yang bisa saya bantu?
Sumber teratas: Bagaimana cara mengetahui apakah suatu produk asli atau palsu?
---
Query: Bagaimana jika penjual menolak pengajuan pengembalian saya?
Jawaban: Halo! Mengenai hal tersebut, Anda dapat mengajukan banding melalui pusat resolusi dengan menyertakan bukti tambahan agar ditinjau oleh tim mediasi. 

Ada hal lain yang bisa saya bantu?
Sumber teratas: Bagaimana jika penjual menolak pengajuan pengembalian saya?
---
Query: Saya memiliki voucher, bagaimana cara menggunakannya?
Jawaban: Halo! Untuk menggunakan kode voucher, kamu bisa memasukkannya pada kolom 'Kode Promo' di halaman checkout sebelum melakukan pembayaran, dan diskon akan otomatis terpotong. 

Ada hal lain ya

In [22]:
# TEST QUERIES 2

test_queries = [
    # TODO: isi minimal 5 query uji coba di sini
    "Apakah bisa menukar barang dengan barang yang berbeda?", # Apakah bisa menukar barang dengan ukuran/warna lain daripada refund?
    "Apakah bisa barang sampai besok",
    "Saya sudah tidak mau menggunakan aplikasi ini lagi, cara hapus akun ini?",
    "Cara mengganti password akun saya yang lupa? berapa 5 + 4? dan mengubah nomor telepon?",
    "Apakah data saya aman? dan metode atau cara apa tang digunakan untuk menjaga keamanan data saya?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: Apakah bisa menukar barang dengan barang yang berbeda?
Jawaban: Halo! Mohon maaf, saya belum memiliki informasi tersebut di dalam FAQ kami. Silakan menghubungi customer service kami untuk mendapatkan bantuan lebih lanjut terkait penukaran dengan barang yang berbeda. Ada hal lain yang bisa saya bantu?
Sumber teratas: Apakah bisa menukar barang dengan ukuran/warna lain daripada refund?
---
Query: Apakah bisa barang sampai besok
Jawaban: Halo! Mohon maaf, saya belum memiliki informasi tersebut. Silakan hubungi customer service kami untuk bantuan lebih lanjut ya.
Sumber teratas: Berapa lama estimasi waktu pengiriman barang?
---
Query: Saya sudah tidak mau menggunakan aplikasi ini lagi, cara hapus akun ini?
Jawaban: Halo! Pengajuan penghapusan akun secara permanen dapat dilakukan melalui menu 'Pengaturan Akun > Hapus Akun' atau menghubungi customer service. 

Ada hal lain yang bisa saya bantu?
Sumber teratas: Bagaimana cara menghapus akun secara permanen?
---
Query: Cara mengganti pa

### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._

# Hasil Pengujian (*Answer*)

Berikut ini merupakan hasil pengujian sistem berdasarkan query yang diberikan. Pengujian dilakukan untuk mengevaluasi kemampuan *retriever* dalam menemukan informasi yang relevan dari *knowledge base* serta kemampuan model dalam menghasilkan jawaban yang sesuai.

## 1. Query 1

### 1.1. Bagaimana cara mengetahui apakah suatu produk asli atau palsu?

* **Asal pertanyaan:** Pertanyaan ini sesuai dengan salah satu entri dalam *knowledge base*, yaitu (42, Produk, “Bagaimana cara mengetahui apakah suatu produk asli atau palsu?”).
* **Jawaban:** Inti jawaban yang dihasilkan sesuai dengan informasi yang terdapat dalam *knowledge base*.
* **Evaluasi:** **Retriever bekerja dengan baik.**

### 1.2. Bagaimana jika penjual menolak pengajuan pengembalian saya?

* **Asal pertanyaan:** Pertanyaan ini sesuai dengan salah satu entri dalam *knowledge base*, yaitu (29, Pengembalian, “Bagaimana jika penjual menolak pengajuan pengembalian saya?”).
* **Jawaban:** Inti jawaban yang dihasilkan sesuai dengan informasi yang terdapat dalam *knowledge base*.
* **Evaluasi:** **Retriever bekerja dengan baik.**

### 1.3. Saya memiliki voucher, bagaimana cara menggunakannya?

* **Asal pertanyaan:** Pertanyaan ini merujuk pada entri dalam *knowledge base*, yaitu (37, Promo, “Bagaimana cara menggunakan kode voucher saat checkout?”). Meskipun susunan kalimatnya berbeda, inti pertanyaan tetap sama.
* **Jawaban:** Inti jawaban yang dihasilkan sesuai dengan informasi yang terdapat dalam *knowledge base*.
* **Evaluasi:** **Retriever bekerja dengan baik.**

### 1.4. Cara mengganti password akun saya yang lupa?

* **Asal pertanyaan:** Pertanyaan ini merujuk pada entri dalam *knowledge base*, yaitu (32, Akun, “Saya lupa password akun, bagaimana cara mengaturnya ulang?”). Pertanyaan disampaikan dengan susunan kalimat yang berbeda, tetapi memiliki maksud yang sama.
* **Jawaban:** Inti jawaban yang dihasilkan sesuai dengan informasi yang terdapat dalam *knowledge base*.
* **Evaluasi:** **Retriever bekerja dengan baik.**

### 1.5. Apa itu *reverse engineering* dan bagaimana cara kerjanya?

* **Asal pertanyaan:** Pertanyaan ini tidak ditemukan dalam *knowledge base*.
* **Jawaban:** Model memberikan respons yang sesuai dengan kondisi tersebut, yaitu menyatakan bahwa informasi yang dibutuhkan tidak diketahui atau tidak tersedia, alih-alih memberikan jawaban yang tidak didukung oleh sumber.
* **Evaluasi:** **Retriever bekerja dengan baik**, dengan catatan bahwa kemampuan model menolak menjawab tanpa sumber yang relevan juga perlu dievaluasi secara terpisah.

## 2. Query 2

### 2.1. Apakah bisa menukar barang dengan barang yang berbeda?

* **Asal pertanyaan:** Pertanyaan ini tidak ditemukan dalam *knowledge base*.
* **Jawaban:** Model memberikan respons yang sesuai dengan kondisi tersebut, yaitu menyatakan bahwa informasi yang dibutuhkan tidak tersedia, alih-alih memberikan jawaban tanpa dasar yang jelas.
* **Evaluasi:** **Retriever bekerja dengan baik**, dengan catatan bahwa respons model juga perlu dievaluasi untuk memastikan tidak terjadi halusinasi.

### 2.2. Apakah bisa barang sampai besok?

* **Asal pertanyaan:** Pertanyaan ini tidak ditemukan dalam *knowledge base*.
* **Jawaban:** Model memberikan respons yang sesuai dengan kondisi tersebut, yaitu menyatakan bahwa informasi yang dibutuhkan tidak tersedia, alih-alih memberikan jawaban tanpa dasar yang jelas.
* **Evaluasi:** **Retriever bekerja dengan baik**, dengan catatan bahwa respons model juga perlu dievaluasi untuk memastikan tidak terjadi halusinasi.

### 2.3. Saya sudah tidak mau menggunakan aplikasi ini lagi, bagaimana cara menghapus akun?

* **Asal pertanyaan:** Pertanyaan ini merujuk pada entri dalam *knowledge base*, yaitu (35, Akun, “Bagaimana cara menghapus akun secara permanen?”). Meskipun susunan kalimatnya berbeda, inti pertanyaan tetap sama.
* **Jawaban:** Inti jawaban yang dihasilkan sesuai dengan informasi yang terdapat dalam *knowledge base*.
* **Evaluasi:** **Retriever bekerja dengan baik.**

### 2.4. Cara mengganti password akun saya yang lupa? Berapa 5 + 4? Dan bagaimana cara mengubah nomor telepon?

* **Asal pertanyaan:** Pertanyaan ini menggabungkan tiga pertanyaan, yaitu dua pertanyaan yang sesuai dengan *knowledge base* dan satu pertanyaan yang tidak berkaitan dengan informasi dalam *knowledge base*. Dua pertanyaan yang relevan merujuk pada entri berikut:

  1. (32, Akun, “Saya lupa password akun, bagaimana cara mengaturnya ulang?”).
  2. (33, Akun, “Bagaimana cara mengubah nomor telepon yang terdaftar di akun?”).

  Sementara itu, pertanyaan “Berapa 5 + 4?” tidak berasal dari *knowledge base*.
* **Jawaban:** Model berhasil menjawab kedua pertanyaan yang informasinya tersedia dalam *knowledge base*, yaitu pertanyaan tentang pengaturan ulang kata sandi dan perubahan nomor telepon. Model juga berhasil menjawab pertanyaan aritmetika sederhana, yaitu 5 + 4 = 9, meskipun pertanyaan tersebut tidak berasal dari *knowledge base*.
* **Evaluasi:** **Retriever bekerja dengan baik** dalam menemukan informasi yang relevan untuk dua pertanyaan yang bersumber dari *knowledge base*. Selain itu, model menunjukkan kemampuan untuk menjawab pertanyaan tambahan di luar cakupan *knowledge base*.

### 2.5. Apakah data saya aman? Metode atau cara apa yang digunakan untuk menjaga keamanan data saya?

* **Asal pertanyaan:** Pertanyaan ini tidak ditemukan dalam *knowledge base*.
* **Jawaban:** Model memberikan respons yang sesuai dengan kondisi tersebut, yaitu menyatakan bahwa informasi yang dibutuhkan tidak tersedia, alih-alih memberikan penjelasan yang tidak didukung oleh sumber.
* **Evaluasi:** **Retriever bekerja dengan baik**, dengan catatan bahwa respons model perlu dievaluasi secara terpisah untuk memastikan model tidak menghasilkan informasi yang tidak didukung oleh *knowledge base*.

## Kesimpulan

Berdasarkan hasil pengujian, sistem menunjukkan kemampuan yang baik dalam menangani pertanyaan yang informasinya tersedia dalam *knowledge base*, termasuk pertanyaan yang disampaikan dengan susunan kalimat berbeda. Sistem juga menunjukkan kemampuan untuk menangani pertanyaan yang tidak tercakup dalam *knowledge base* dengan tidak memberikan jawaban yang tidak didukung oleh sumber.

Selain itu, pada pengujian yang menggabungkan beberapa pertanyaan dalam satu query, model mampu menjawab pertanyaan yang relevan dengan *knowledge base* serta pertanyaan tambahan di luar cakupan sumber.

Meskipun demikian, evaluasi perlu membedakan kinerja *retriever* dalam menemukan dokumen yang relevan dengan kemampuan model dalam menyusun jawaban dan menolak menjawab ketika informasi tidak tersedia. Oleh karena itu, hasil pengujian ini menunjukkan indikasi kinerja yang baik, tetapi belum cukup untuk menyimpulkan bahwa seluruh komponen sistem telah bekerja secara optimal tanpa pengujian tambahan.
